# Precious-Metals Statistical Arbitrage

## 1. Research question and decision standard

This notebook asks whether economically related precious-metals instruments exhibit a repeatable divergence/convergence effect that remains tradable after costs. It may conclude that no useful statistical arbitrage exists. Statistical significance is necessary, not sufficient, and none of the results guarantees future profit.

Verdicts are restricted to **Robust statistical-arbitrage evidence**, **Interesting but weak evidence**, or **No useful statistical arbitrage**.

## 2. Imports, deterministic configuration, and assumptions

Every historical estimate is point-in-time. The signal is observed first, execution occurs one session later, and P&L starts after execution. The base model charges 5 basis points per dollar of one-way turnover and 2% annualized short borrow.

The workflow follows the practical cautions in Michael Isichenko, *Quantitative Portfolio Management* (Wiley, 2021): prefer economically motivated relationships to brute-force pair mining, translate residual forecasts into actual tradable legs, model point-in-time information and costs, and avoid repeated P&L-driven tuning.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Callable
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.stattools import adfuller, coint

from ats.ticker import EquityTicker

RANDOM_SEED = 20260926
TRADABLE_TICKERS = ("GLD", "IAU", "SLV", "GDX", "GDXJ", "SIL", "SILJ")
FACTOR_TICKERS = ("SPY", "UUP", "IEF", "TIP")


@dataclass(frozen=True)
class ResearchConfig:
    discovery_fraction: float = 0.60
    validation_fraction: float = 0.20
    entry_z: float = 2.0
    exit_z: float = 0.5
    max_holding_days: int = 20
    transaction_cost_bps: float = 5.0
    annual_short_borrow: float = 0.02
    bootstrap_repetitions: int = 2_000
    bootstrap_block_length: int = 40
    horizons: tuple[int, ...] = (1, 5, 10, 20, 40)
    estimation_windows: tuple[int, ...] = (252, 504)
    accumulation_windows: tuple[int, ...] = (5, 10, 20)
    normalization_windows: tuple[int, ...] = (126, 252)
    robustness_entry_z: tuple[float, ...] = (1.5, 2.0, 2.5)
    robustness_exit_z: tuple[float, ...] = (0.0, 0.5)
    robustness_holding_days: tuple[int, ...] = (10, 20, 40)
    robustness_cost_bps: tuple[float, ...] = (2.0, 5.0, 10.0)
    robustness_borrow: tuple[float, ...] = (0.0, 0.02, 0.05)


@dataclass(frozen=True)
class SplitBoundaries:
    discovery_end: pd.Timestamp
    validation_end: pd.Timestamp
    test_end: pd.Timestamp


CONFIG = ResearchConfig()
np.random.seed(RANDOM_SEED)


def validate_price_panel(prices: pd.DataFrame) -> pd.DataFrame:
    '''Return sorted positive finite prices without inventing observations.'''
    if prices.index.has_duplicates:
        duplicates = prices.index[prices.index.duplicated()].unique()
        raise ValueError(f"duplicate price dates: {duplicates.tolist()}")
    clean = prices.copy()
    clean.index = pd.DatetimeIndex(clean.index)
    clean = clean.sort_index().astype(float)
    return clean.where(np.isfinite(clean) & clean.gt(0.0))


def simple_and_log_returns(prices: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Compute close-to-close returns while preserving gaps as missing.'''
    clean = validate_price_panel(prices)
    observed_pair = clean.notna() & clean.shift(1).notna()
    simple = clean.pct_change(fill_method=None).where(observed_pair)
    log = np.log(clean).diff().where(observed_pair)
    return simple, log


def fetch_price_panel(
    tickers: tuple[str, ...],
    ticker_factory: Callable[..., EquityTicker] = EquityTicker,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Fetch maximum ATS histories and return a wide panel plus an audit table.'''
    series: dict[str, pd.Series] = {}
    audit_rows: list[dict[str, Any]] = []
    for symbol in tickers:
        try:
            ticker = ticker_factory(symbol).fetch_price_data(
                all_available_price_history=True
            )
            frame = ticker.price_data.select("date", "close").to_pandas()
            frame["date"] = pd.to_datetime(frame["date"])
            if frame.empty:
                raise ValueError("empty download")
            if frame["date"].duplicated().any():
                raise ValueError("duplicate dates")
            values = pd.Series(
                frame["close"].to_numpy(dtype=float),
                index=pd.DatetimeIndex(frame["date"]),
                name=symbol,
            ).sort_index()
            values = values.where(np.isfinite(values) & values.gt(0.0))
            series[symbol] = values
            audit_rows.append(
                {
                    "symbol": symbol,
                    "status": "ok",
                    "first_date": values.first_valid_index(),
                    "last_date": values.last_valid_index(),
                    "observations": int(values.notna().sum()),
                    "missing": int(values.isna().sum()),
                }
            )
        except Exception as exc:
            audit_rows.append(
                {
                    "symbol": symbol,
                    "status": f"excluded: {exc}",
                    "first_date": pd.NaT,
                    "last_date": pd.NaT,
                    "observations": 0,
                    "missing": 0,
                }
            )
    panel = pd.concat(series.values(), axis=1).sort_index() if series else pd.DataFrame()
    return panel, pd.DataFrame(audit_rows).set_index("symbol")


def chronological_boundaries(index: pd.DatetimeIndex) -> SplitBoundaries:
    '''Create ordered 60/20/20 date boundaries once for a candidate sample.'''
    dates = pd.DatetimeIndex(index).drop_duplicates().sort_values()
    if len(dates) < 5:
        raise ValueError("at least five observations are required for chronological splits")
    discovery_count = max(1, int(np.floor(len(dates) * CONFIG.discovery_fraction)))
    validation_count = max(1, int(np.floor(len(dates) * CONFIG.validation_fraction)))
    validation_end_pos = min(discovery_count + validation_count - 1, len(dates) - 2)
    return SplitBoundaries(
        discovery_end=dates[discovery_count - 1],
        validation_end=dates[validation_end_pos],
        test_end=dates[-1],
    )


def rolling_ols_lagged(
    y: pd.Series,
    x: pd.DataFrame,
    window: int,
    ridge_alpha: float = 0.0,
) -> tuple[pd.DataFrame, pd.Series]:
    '''Fit trailing regressions through t-1 and stamp coefficients/residual at t.'''
    if window < 2:
        raise ValueError("window must be at least two")
    aligned = pd.concat([y.rename("__target__"), x], axis=1).sort_index()
    columns = ["const", *x.columns]
    coefficients = pd.DataFrame(np.nan, index=aligned.index, columns=columns)
    residuals = pd.Series(np.nan, index=aligned.index, name="residual")
    penalty = np.diag([0.0, *([1.0] * x.shape[1])])

    for position in range(window, len(aligned)):
        history = aligned.iloc[position - window : position].dropna()
        if len(history) <= len(columns):
            continue
        design = np.column_stack(
            [np.ones(len(history)), history[x.columns].to_numpy(dtype=float)]
        )
        target = history["__target__"].to_numpy(dtype=float)
        condition = np.linalg.cond(design)
        if ridge_alpha > 0.0:
            estimate = np.linalg.solve(
                design.T @ design + ridge_alpha * penalty,
                design.T @ target,
            )
        elif not np.isfinite(condition) or condition > 1e12:
            continue
        else:
            estimate = np.linalg.lstsq(design, target, rcond=None)[0]
        timestamp = aligned.index[position]
        coefficients.loc[timestamp] = estimate
        current = aligned.iloc[position]
        if current.notna().all():
            current_design = np.r_[1.0, current[x.columns].to_numpy(dtype=float)]
            residuals.loc[timestamp] = float(current["__target__"] - current_design @ estimate)
    return coefficients, residuals


def lagged_robust_zscore(values: pd.Series, window: int) -> pd.Series:
    '''Standardize each value from a median and scale ending at t-1.'''
    if window < 3:
        raise ValueError("window must be at least three")
    result = pd.Series(np.nan, index=values.index, name="zscore")
    numeric = values.astype(float)
    for position in range(window, len(numeric)):
        history = numeric.iloc[position - window : position].dropna()
        current = numeric.iloc[position]
        if len(history) < max(3, window // 2) or not np.isfinite(current):
            continue
        center = float(history.median())
        mad = float((history - center).abs().median())
        scale = 1.4826 * mad
        if not np.isfinite(scale) or scale <= 0.0:
            scale = float(history.std(ddof=1))
        if np.isfinite(scale) and scale > 0.0:
            result.iloc[position] = (current - center) / scale
    return result


@dataclass(frozen=True)
class Candidate:
    name: str
    family: str
    spread: pd.Series
    zscore: pd.Series
    leg_weights: pd.DataFrame
    diagnostics: dict[str, Any]


def _normalize_rows(weights: pd.DataFrame) -> pd.DataFrame:
    gross = weights.abs().sum(axis=1).replace(0.0, np.nan)
    return weights.div(gross, axis=0)


def make_cointegration_candidate(
    log_prices: pd.DataFrame,
    left: str,
    right: str,
    estimation_window: int,
    normalization_window: int,
    discovery_mask: pd.Series,
) -> Candidate | None:
    '''Build a lagged rolling Engle-Granger spread for an economic pair.'''
    data = log_prices[[left, right]].dropna().sort_index()
    if len(data) < estimation_window + normalization_window + 20:
        return None
    mask = discovery_mask.reindex(data.index).fillna(False).astype(bool)
    discovery = data.loc[mask]
    if len(discovery) < estimation_window + 20:
        return None
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _, coint_pvalue, _ = coint(discovery[left], discovery[right])
    coefficients, residual = rolling_ols_lagged(
        data[left], data[[right]], estimation_window
    )
    beta = coefficients[right]
    weights = _normalize_rows(pd.DataFrame({left: 1.0, right: -beta}, index=data.index))
    zscore = lagged_robust_zscore(residual, normalization_window)
    beta_discovery = beta.loc[mask].dropna()
    beta_scale = abs(float(beta_discovery.mean()))
    stability = float(beta_discovery.std() / beta_scale) if beta_scale > 1e-12 else np.inf
    return Candidate(
        name=f"coint:{left}~{right}:w{estimation_window}:z{normalization_window}",
        family="cointegration",
        spread=residual.rename("spread"),
        zscore=zscore,
        leg_weights=weights,
        diagnostics={
            "coint_pvalue": float(coint_pvalue),
            "stationarity_ok": bool(np.isfinite(coint_pvalue) and coint_pvalue < 0.05),
            "stable_weights": bool(np.isfinite(stability) and stability < 0.75),
            "weight_variation": stability,
            "coefficients": coefficients,
            "definition": f"log({left}) - alpha[t-1] - beta[t-1] log({right})",
        },
    )


def make_factor_candidate(
    log_returns: pd.DataFrame,
    target: str,
    factors: tuple[str, ...],
    estimation_window: int,
    accumulation_window: int,
    normalization_window: int,
    discovery_mask: pd.Series,
    ridge_alpha: float = 0.0,
) -> Candidate | None:
    '''Build a trailing factor-residual displacement with tradable hedge legs.'''
    columns = [target, *factors]
    data = log_returns[columns].dropna().sort_index()
    if len(data) < estimation_window + normalization_window + accumulation_window:
        return None
    coefficients, residual = rolling_ols_lagged(
        data[target], data[list(factors)], estimation_window, ridge_alpha
    )
    displacement = residual.rolling(accumulation_window, min_periods=accumulation_window).sum()
    zscore = lagged_robust_zscore(displacement, normalization_window)
    raw_weights = pd.DataFrame(index=data.index, columns=columns, dtype=float)
    raw_weights[target] = 1.0
    for factor in factors:
        raw_weights[factor] = -coefficients[factor]
    weights = _normalize_rows(raw_weights)
    mask = discovery_mask.reindex(data.index).fillna(False).astype(bool)
    coefficient_slice = coefficients.loc[mask, list(factors)].dropna()
    variation = float(coefficient_slice.std().mean()) if not coefficient_slice.empty else np.inf
    discovery_residual = residual.loc[mask].dropna()
    stationarity_pvalue = (
        float(adfuller(discovery_residual, autolag="AIC")[1])
        if len(discovery_residual) >= 50
        else np.nan
    )
    return Candidate(
        name=(
            f"factor:{target}~{'+'.join(factors)}:w{estimation_window}:"
            f"k{accumulation_window}:z{normalization_window}"
        ),
        family="factor_residual",
        spread=displacement.rename("spread"),
        zscore=zscore,
        leg_weights=weights,
        diagnostics={
            "stationarity_pvalue": stationarity_pvalue,
            "stationarity_ok": bool(
                np.isfinite(stationarity_pvalue) and stationarity_pvalue < 0.05
            ),
            "stable_weights": bool(np.isfinite(variation) and variation < 0.75),
            "weight_variation": variation,
            "coefficients": coefficients,
            "definition": (
                f"{accumulation_window}-day sum of lagged-OLS {target} residuals "
                f"against {', '.join(factors)}"
            ),
        },
    )


def make_pca_candidates(
    log_returns: pd.DataFrame,
    assets: tuple[str, ...],
    estimation_window: int,
    accumulation_window: int,
    normalization_window: int,
    discovery_mask: pd.Series,
) -> list[Candidate]:
    '''Build rolling PCA reconstruction-residual candidates for a coherent basket.'''
    data = log_returns[list(assets)].dropna().sort_index()
    if len(data) < estimation_window + normalization_window + accumulation_window:
        return []
    residuals = pd.DataFrame(np.nan, index=data.index, columns=assets)
    weight_panels = {
        asset: pd.DataFrame(np.nan, index=data.index, columns=assets) for asset in assets
    }
    explained = pd.Series(np.nan, index=data.index)
    component_count = pd.Series(np.nan, index=data.index)
    for position in range(estimation_window, len(data)):
        history = data.iloc[position - estimation_window : position]
        means = history.mean()
        scales = history.std(ddof=1)
        if (scales <= 0.0).any() or not np.isfinite(scales).all():
            continue
        standardized = (history - means) / scales
        covariance = standardized.cov().to_numpy()
        eigenvalues, eigenvectors = np.linalg.eigh(covariance)
        order = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[order]
        eigenvectors = eigenvectors[:, order]
        ratios = eigenvalues / eigenvalues.sum()
        components = min(2, int(np.searchsorted(np.cumsum(ratios), 0.70) + 1))
        loadings = eigenvectors[:, :components]
        projection = loadings @ loadings.T
        residual_operator = np.eye(len(assets)) - projection
        current = ((data.iloc[position] - means) / scales).to_numpy(dtype=float)
        current_residual = residual_operator @ current
        timestamp = data.index[position]
        residuals.loc[timestamp] = current_residual
        explained.loc[timestamp] = float(ratios[:components].sum())
        component_count.loc[timestamp] = components
        return_operator = residual_operator @ np.diag(1.0 / scales.to_numpy())
        for asset_position, asset in enumerate(assets):
            weight_panels[asset].loc[timestamp] = return_operator[asset_position]

    candidates: list[Candidate] = []
    mask = discovery_mask.reindex(data.index).fillna(False).astype(bool)
    for asset in assets:
        displacement = residuals[asset].rolling(
            accumulation_window, min_periods=accumulation_window
        ).sum()
        zscore = lagged_robust_zscore(displacement, normalization_window)
        weights = _normalize_rows(weight_panels[asset])
        discovered = displacement.loc[mask].dropna()
        stationarity_pvalue = (
            float(adfuller(discovered, autolag="AIC")[1]) if len(discovered) >= 50 else np.nan
        )
        candidates.append(
            Candidate(
                name=(
                    f"pca:{asset}|{'+'.join(assets)}:w{estimation_window}:"
                    f"k{accumulation_window}:z{normalization_window}"
                ),
                family="pca_residual",
                spread=displacement.rename("spread"),
                zscore=zscore,
                leg_weights=weights,
                diagnostics={
                    "stationarity_pvalue": stationarity_pvalue,
                    "stationarity_ok": bool(
                        np.isfinite(stationarity_pvalue) and stationarity_pvalue < 0.05
                    ),
                    "stable_weights": True,
                    "explained_variance": explained,
                    "component_count": component_count,
                    "definition": (
                        f"{accumulation_window}-day rolling-PCA reconstruction residual "
                        f"for {asset} in {', '.join(assets)}"
                    ),
                },
            )
        )
    return candidates

In [ ]:
def _split_dates(
    index: pd.DatetimeIndex,
    split: str,
    boundaries: SplitBoundaries,
) -> tuple[pd.Timestamp, pd.Timestamp]:
    if split == "discovery":
        return index.min(), boundaries.discovery_end
    if split == "validation":
        return boundaries.discovery_end, boundaries.validation_end
    if split == "test":
        return boundaries.validation_end, boundaries.test_end
    raise ValueError(f"unknown split: {split}")


def purged_forward_convergence(
    candidate: Candidate,
    horizons: tuple[int, ...],
    split: str,
    boundaries: SplitBoundaries,
) -> pd.DataFrame:
    '''Create signed forward convergence labels that cannot cross a split boundary.'''
    base = pd.concat(
        [candidate.spread.rename("spread"), candidate.zscore.rename("zscore")], axis=1
    ).sort_index()
    start, end = _split_dates(base.index, split, boundaries)
    rows = []
    dates = pd.Series(base.index, index=base.index)
    for horizon in horizons:
        future_spread = base["spread"].shift(-horizon)
        end_date = dates.shift(-horizon)
        convergence = -np.sign(base["zscore"]) * (future_spread - base["spread"])
        frame = pd.DataFrame(
            {
                "candidate": candidate.name,
                "horizon": horizon,
                "end_date": end_date,
                "zscore": base["zscore"],
                "abs_z": base["zscore"].abs(),
                "convergence": convergence,
            },
            index=base.index,
        )
        if split == "discovery":
            allowed = (frame.index >= start) & (frame.index <= end)
        else:
            allowed = (frame.index > start) & (frame.index <= end)
        allowed &= frame["end_date"].notna() & (frame["end_date"] <= end)
        rows.append(frame.loc[allowed].dropna())
    return pd.concat(rows).sort_index() if rows else pd.DataFrame()


def nonoverlapping_extreme_events(
    evidence: pd.DataFrame,
    horizon: int,
    threshold: float,
) -> pd.DataFrame:
    '''Greedily retain extreme events separated by at least horizon observations.'''
    ordered = evidence.sort_index()
    positions = np.flatnonzero(ordered["abs_z"].to_numpy() >= threshold)
    retained: list[int] = []
    last = -horizon
    for position in positions:
        if position - last >= horizon:
            retained.append(int(position))
            last = int(position)
    return ordered.iloc[retained]


def moving_block_mean_test(
    values: pd.Series,
    repetitions: int,
    block_length: int,
    seed: int,
) -> dict[str, float]:
    '''Return a moving-block mean interval and one-sided positive-mean p-value.'''
    sample = values.dropna().to_numpy(dtype=float)
    if len(sample) < 3:
        return {"mean": np.nan, "ci_low": np.nan, "ci_high": np.nan, "pvalue": np.nan}
    block = max(1, min(int(block_length), len(sample)))
    rng = np.random.default_rng(seed)
    starts = np.arange(len(sample))
    boot_means = np.empty(repetitions)
    null_means = np.empty(repetitions)
    centered = sample - sample.mean()
    blocks_needed = int(np.ceil(len(sample) / block))
    offsets = np.arange(block)
    for repetition in range(repetitions):
        chosen = rng.choice(starts, size=blocks_needed, replace=True)
        indices = ((chosen[:, None] + offsets) % len(sample)).ravel()[: len(sample)]
        boot_means[repetition] = sample[indices].mean()
        null_means[repetition] = centered[indices].mean()
    observed = float(sample.mean())
    pvalue = float((1 + np.sum(null_means >= observed)) / (repetitions + 1))
    return {
        "mean": observed,
        "ci_low": float(np.quantile(boot_means, 0.025)),
        "ci_high": float(np.quantile(boot_means, 0.975)),
        "pvalue": pvalue,
    }


def holm_adjust(pvalues: pd.Series) -> pd.Series:
    '''Apply Holm family-wise error control while preserving labels.'''
    numeric = pvalues.astype(float)
    adjusted = pd.Series(np.nan, index=numeric.index, dtype=float)
    finite = numeric.notna()
    if finite.any():
        adjusted.loc[finite] = multipletests(numeric.loc[finite], method="holm")[1]
    return adjusted


def discovery_evidence(
    candidates: list[Candidate],
    boundaries: SplitBoundaries,
    config: ResearchConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Evaluate discovery convergence shape and apply family-wise promotion gates.'''
    summaries: list[dict[str, Any]] = []
    bucket_frames: list[pd.DataFrame] = []
    for candidate in candidates:
        evidence = purged_forward_convergence(
            candidate, config.horizons, "discovery", boundaries
        )
        for horizon in config.horizons:
            horizon_data = evidence.loc[evidence["horizon"] == horizon].copy()
            if horizon_data.empty:
                continue
            try:
                horizon_data["bucket"] = pd.qcut(
                    horizon_data["abs_z"], 5, labels=False, duplicates="drop"
                )
            except ValueError:
                horizon_data["bucket"] = 0
            buckets = (
                horizon_data.groupby("bucket", dropna=False)
                .agg(
                    observations=("convergence", "size"),
                    mean_abs_z=("abs_z", "mean"),
                    mean_convergence=("convergence", "mean"),
                    median_convergence=("convergence", "median"),
                )
                .reset_index()
            )
            buckets.insert(0, "horizon", horizon)
            buckets.insert(0, "candidate", candidate.name)
            bucket_frames.append(buckets)
            regression_data = horizon_data[["abs_z", "convergence"]].dropna()
            if len(regression_data) >= 10 and regression_data["abs_z"].std() > 0:
                design = sm.add_constant(regression_data["abs_z"])
                fitted = sm.OLS(regression_data["convergence"], design).fit(
                    cov_type="HAC", cov_kwds={"maxlags": horizon}
                )
                slope = float(fitted.params["abs_z"])
                slope_pvalue = float(fitted.pvalues["abs_z"] / 2) if slope > 0 else 1.0
            else:
                slope, slope_pvalue = np.nan, 1.0
            events = nonoverlapping_extreme_events(
                horizon_data, horizon, config.entry_z
            )
            bootstrap = moving_block_mean_test(
                events["convergence"],
                config.bootstrap_repetitions,
                config.bootstrap_block_length,
                RANDOM_SEED + horizon,
            )
            raw_pvalue = max(slope_pvalue, bootstrap["pvalue"])
            summaries.append(
                {
                    "candidate": candidate.name,
                    "family": candidate.family,
                    "horizon": horizon,
                    "slope": slope,
                    "event_count": len(events),
                    "mean_convergence": bootstrap["mean"],
                    "ci_low": bootstrap["ci_low"],
                    "ci_high": bootstrap["ci_high"],
                    "raw_pvalue": raw_pvalue,
                    "stationarity_ok": bool(
                        candidate.diagnostics.get("stationarity_ok", False)
                    ),
                    "stable_weights": bool(
                        candidate.diagnostics.get("stable_weights", False)
                    ),
                }
            )
    summary = pd.DataFrame(summaries)
    if summary.empty:
        return summary, pd.concat(bucket_frames, ignore_index=True) if bucket_frames else pd.DataFrame()
    summary["adjusted_pvalue"] = holm_adjust(summary["raw_pvalue"])
    summary["passes"] = (
        summary["stationarity_ok"]
        & summary["stable_weights"]
        & summary["slope"].gt(0.0)
        & summary["event_count"].ge(30)
        & summary["adjusted_pvalue"].lt(0.05)
    )
    summary["rejection_reason"] = np.select(
        [
            ~summary["stationarity_ok"],
            ~summary["stable_weights"],
            summary["event_count"].lt(30),
            summary["slope"].le(0.0) | summary["slope"].isna(),
            summary["adjusted_pvalue"].ge(0.05),
        ],
        [
            "stationarity gate failed",
            "hedge instability",
            "fewer than 30 non-overlapping events",
            "nonpositive convergence slope",
            "Holm-adjusted p-value is not below 0.05",
        ],
        default="passes discovery gates",
    )
    buckets = pd.concat(bucket_frames, ignore_index=True) if bucket_frames else pd.DataFrame()
    return summary, buckets

## 3. ATS API audit and data acquisition

Data are fetched below through `EquityTicker`; adjusted closes and actual observation dates are retained.

## 4. Data quality and chronological splits

Each candidate uses chronological 60/20/20 discovery, validation, and untouched-test partitions.

## 5. Reusable leakage-safe research functions

The tagged cells below contain the estimators, inference, and backtest machinery exercised by focused tests.

## 6. Deterministic self-tests

Assertions here complement the focused pytest suite with notebook-visible invariants.

## 7. Discovery diagnostics for candidate families

Only economically constrained cointegration, factor-residual, and rolling-PCA candidates are considered.

## 8. Convergence shape, robust inference, and rejection table

Forward convergence is tested at several horizons with overlap-aware inference and discovery-family multiplicity control.

## 9. Frozen strategy specification

The base entry, exit, holding period, sizing, timing, and cost assumptions are frozen before the untouched test is exposed.

## 10. Validation results and local robustness

Validation can reject the discovery candidate; robustness is diagnostic and never feeds parameter selection.

## 11. Untouched out-of-sample test

At most one validated primary candidate reaches this section.

## 12. Trading performance, regimes, and failure modes

Gross and net results, turnover, borrow, tails, drawdowns, subperiods, and lagged regimes are reported separately.

## 13. Concise final research summary

The executed notebook ends with the exact divergence, evidence, rule, out-of-sample result, robustness, limitations, and permitted verdict.